# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_1color

This notebook builds the first scoped submission for the `fill_enclosed_regions / nonlocal_1color` subtype.

Workflow:

1. Load the strict 41-task subtype from `task_type_map.csv`.
2. Try bounded wider additive-template models for tasks where 3x3 context is insufficient.
3. Fall back to compact additive candidates when they visibly fit.
4. Emit identity fallback models for any remaining task so the submission zip is complete.
5. Build `/kaggle/working/submission.zip` from exactly this subtype scope.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None




"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""


BATCH, CH, H, W = 1, 10, 30, 30


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_initializer(name, array):
    arr = np.asarray(array, dtype=np.float16)
    return numpy_helper.from_array(arr, name=name)


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong_expected = None
    first_wrong_actual =None
    print("len of examples: ", len(examples))

    count_ex =1
    for ex in examples:
        if count_ex % 10 ==0:
            print(f"example {count_ex}")
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong_expected is None:
                first_wrong_expected = ex
                first_wrong_actual = actual

        count_ex = count_ex +1
        
    return {"right": right, "wrong": wrong, 
            "first_wrong_expected": first_wrong_expected, "first_wrong_actual":first_wrong_actual}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path


In [2]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but build_family_submission
# must import onnx to create taskNNN.onnx files.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing ONNX packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


import onnx
import onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 69.7 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:
from pathlib import Path
import ast
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
SUBTYPE = 'nonlocal_1color'
MODEL_VERSION = 'fill-additive-nonlocal-1color-v0.75-task341-static-bridge'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / f'{FAMILY}_{SUBTYPE}_task341_static_bridge_v75'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)


DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task341_static_bridge_v75
MODEL_VERSION = fill-additive-nonlocal-1color-v0.75-task341-static-bridge


In [4]:
# update for each task
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()


def parse_color_list(value):
    if pd.isna(value) or value == '':
        return []
    if isinstance(value, list):
        return value
    try:
        return list(ast.literal_eval(str(value)))
    except Exception:
        return []

family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
nonlocal_1color_df = family_df[family_df['task_id'].isin(['task341'])].copy().reset_index(drop=True)
task_ids = ['task341']

print('family:', FAMILY)
print('subtype:', SUBTYPE)
print('family tasks:', len(family_df))
print('selected task341 static bridge tasks:', len(task_ids))
print(task_ids)
display(nonlocal_1color_df.head(10))


family: fill_enclosed_regions
subtype: nonlocal_1color
family tasks: 59
selected task341 static bridge tasks: 1
['task341']


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes,parsed_new_output_colors
0,task341,341,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,2679,NaN,0.889833,661,6000,1.0,2679,2679,Same shape; input is mostly preserved while ne...,[8]


In [5]:
# Inspect one scoped task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this subtype.')

task341 examples: 266
first input shape: (10, 10)
first output shape: (10, 10)
first input: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 0, 0, 0, 0]]
first output: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 8, 8, 0, 0, 0, 0, 0, 0], [0, 0, 8, 8, 0, 0, 0, 0, 0, 0], [0, 0, 8, 8, 0, 0, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 0, 0, 0, 0]]


In [6]:
# nonlocal_1color selection table: these are the tasks this notebook version is responsible for.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'local_3x3_score',
    'local_3x3_conflicts',
    'added_nonzero_cells',
    'candidate_flags',
]
fill_selection = nonlocal_1color_df[selection_cols].reset_index(drop=True)
print('selected nonlocal_1color fill/additive tasks:', len(fill_selection))
display(fill_selection)

selected nonlocal_1color fill/additive tasks: 1


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,local_3x3_score,local_3x3_conflicts,added_nonzero_cells,candidate_flags
0,task341,medium,3,1,262,same_shape_variable_size,10x10:266,10x10:266,"[0,1,2,3,4,5,6,7,9]","[0,1,2,3,4,5,6,7,8,9]",[8],0.889833,661,2679,adds_new_color_preserves_input|new_output_colors


In [7]:
# task sepcific modelling modules

CLEAR = 10
ZERO_HOT = -1
NO_CHANGE = -2


def input_canvas(grid):
    canvas = np.full((H, W), CLEAR, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def output_canvas(grid):
    canvas = np.full((H, W), ZERO_HOT, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def task_color_sets(task):
    in_colors = sorted({int(v) for ex in all_examples(task) for row in ex['input'] for v in row})
    out_colors = sorted({int(v) for ex in all_examples(task) for row in ex['output'] for v in row})
    return in_colors, out_colors


def exact_grid_match(task, predictor):
    right = 0
    total = 0
    first_wrong = None
    for split in ['train', 'test', 'arc-gen']:
        for idx, ex in enumerate(task.get(split, [])):
            total += 1
            pred = predictor(ex['input'])
            if pred == ex['output']:
                right += 1
            elif first_wrong is None:
                first_wrong = (split, idx)
    return right, total, first_wrong



def evaluate_predictor_on_task(task, predictor):
    right = 0
    total = 0
    first_wrong = None
    split_counts = {}
    for split in ['train', 'test', 'arc-gen']:
        split_right = 0
        split_total = 0
        for idx, ex in enumerate(task.get(split, [])):
            split_total += 1
            total += 1
            pred = predictor(ex['input'])
            if pred == ex['output']:
                split_right += 1
                right += 1
            elif first_wrong is None:
                first_wrong = f'{split}[{idx}]'
        split_counts[f'{split}_right'] = split_right
        split_counts[f'{split}_total'] = split_total
    return {
        'right': right,
        'total': total,
        'accuracy': right / total if total else None,
        'first_wrong': first_wrong,
        **split_counts,
    }

from collections import defaultdict

# --- v75 task341 static bridge override ---------------------------------------
# Mark color 8 on zero cells that are horizontally or vertically between nonzero
# cells, after 3-cell erosion in the perpendicular direction. Direct ONNX graph.


def simulate_task341_static_bridge(input_grid):
    arr = np.asarray(input_grid, dtype=np.int64)
    h, w = arr.shape
    nonzero = arr != 0
    zero = arr == 0
    horiz = np.zeros_like(zero, dtype=bool)
    for r in range(h):
        left_seen = np.maximum.accumulate(nonzero[r])
        right_seen = np.maximum.accumulate(nonzero[r][::-1])[::-1]
        horiz[r] = left_seen & right_seen & zero[r]
    horiz_eroded = np.zeros_like(horiz)
    for r in range(1, h - 1):
        horiz_eroded[r] = horiz[r - 1] & horiz[r] & horiz[r + 1]
    vert = np.zeros_like(zero, dtype=bool)
    for c in range(w):
        up_seen = np.maximum.accumulate(nonzero[:, c])
        down_seen = np.maximum.accumulate(nonzero[::-1, c])[::-1]
        vert[:, c] = up_seen & down_seen & zero[:, c]
    vert_eroded = np.zeros_like(vert)
    for c in range(1, w - 1):
        vert_eroded[:, c] = vert[:, c - 1] & vert[:, c] & vert[:, c + 1]
    out = arr.copy()
    out[(horiz_eroded | vert_eroded) & zero] = 8
    return out.tolist()


def fit_task341_static_bridge(task):
    right, total, first_wrong = exact_grid_match(task, simulate_task341_static_bridge)
    return {'kind': 'task341_static_bridge'}, {
        'ok': right == total,
        'trainer': 'task341_static_bridge_direct_onnx',
        'model_version': MODEL_VERSION,
        'semantic_kind': 'eroded_horizontal_vertical_nonzero_bridge_mark_color8',
        'visible_right': right,
        'visible_total': total,
        'first_wrong': first_wrong,
        'estimated_model_size_bytes': 9020,
    }


def make_task341_static_bridge_model(payload=None):
    require_onnx()
    nodes, inits = [], []

    def init_arr(name, arr):
        inits.append(numpy_helper.from_array(np.asarray(arr), name))
        return name

    inputs = [helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, 10, 30, 30])]
    outputs = [helper.make_tensor_value_info('output', TensorProto.FLOAT, [1, 10, 30, 30])]
    init_arr('s0', np.array([0], np.int64))
    init_arr('e1', np.array([1], np.int64))
    init_arr('s1', np.array([1], np.int64))
    init_arr('e10', np.array([10], np.int64))
    init_arr('axc', np.array([1], np.int64))
    init_arr('st1', np.array([1], np.int64))
    nodes.append(helper.make_node('Slice', ['input', 's0', 'e1', 'axc', 'st1'], ['zero'], 'slice_zero'))
    nodes.append(helper.make_node('Slice', ['input', 's1', 'e10', 'axc', 'st1'], ['nonzero_channels'], 'slice_nonzero_channels'))
    init_arr('k_nz', np.ones((1, 9, 1, 1), np.float32))
    nodes.append(helper.make_node('Conv', ['nonzero_channels', 'k_nz'], ['nz'], 'conv_nonzero_sum'))
    init_arr('halff', np.array(0.5, np.float32))
    init_arr('threef', np.array(3.0, np.float32))
    nodes.append(helper.make_node('Greater', ['zero', 'halff'], ['zero_bool'], 'zero_bool'))
    init_arr('Tlower', np.tril(np.ones((30, 30), np.float32)))
    init_arr('Tupper', np.triu(np.ones((30, 30), np.float32)))
    nodes.append(helper.make_node('MatMul', ['nz', 'Tlower'], ['left_count'], 'left_count'))
    nodes.append(helper.make_node('MatMul', ['nz', 'Tupper'], ['right_count'], 'right_count'))
    nodes.append(helper.make_node('Greater', ['left_count', 'halff'], ['left_seen'], 'left_seen'))
    nodes.append(helper.make_node('Greater', ['right_count', 'halff'], ['right_seen'], 'right_seen'))
    nodes.append(helper.make_node('And', ['left_seen', 'right_seen'], ['lr'], 'lr'))
    nodes.append(helper.make_node('And', ['lr', 'zero_bool'], ['horiz'], 'horiz'))
    init_arr('k3v', np.ones((1, 1, 3, 1), np.float32))
    init_arr('k3h', np.ones((1, 1, 1, 3), np.float32))
    nodes.append(helper.make_node('Cast', ['horiz'], ['horiz_f'], 'cast_horiz', to=TensorProto.FLOAT))
    nodes.append(helper.make_node('Conv', ['horiz_f', 'k3v'], ['hconv'], 'hconv', pads=[1, 0, 1, 0]))
    nodes.append(helper.make_node('Equal', ['hconv', 'threef'], ['herode'], 'herode'))
    nodes.append(helper.make_node('MatMul', ['Tlower', 'nz'], ['up_count'], 'up_count'))
    nodes.append(helper.make_node('MatMul', ['Tupper', 'nz'], ['down_count'], 'down_count'))
    nodes.append(helper.make_node('Greater', ['up_count', 'halff'], ['up_seen'], 'up_seen'))
    nodes.append(helper.make_node('Greater', ['down_count', 'halff'], ['down_seen'], 'down_seen'))
    nodes.append(helper.make_node('And', ['up_seen', 'down_seen'], ['ud'], 'ud'))
    nodes.append(helper.make_node('And', ['ud', 'zero_bool'], ['vert'], 'vert'))
    nodes.append(helper.make_node('Cast', ['vert'], ['vert_f'], 'cast_vert', to=TensorProto.FLOAT))
    nodes.append(helper.make_node('Conv', ['vert_f', 'k3h'], ['vconv'], 'vconv', pads=[0, 1, 0, 1]))
    nodes.append(helper.make_node('Equal', ['vconv', 'threef'], ['verode'], 'verode'))
    nodes.append(helper.make_node('Or', ['herode', 'verode'], ['bridge'], 'bridge'))
    nodes.append(helper.make_node('And', ['bridge', 'zero_bool'], ['mark_bool'], 'mark'))
    nodes.append(helper.make_node('Not', ['mark_bool'], ['not_mark_bool'], 'not_mark'))
    nodes.append(helper.make_node('Cast', ['not_mark_bool'], ['not_mark'], 'cast_not', to=TensorProto.FLOAT))
    nodes.append(helper.make_node('Cast', ['mark_bool'], ['mark_f'], 'cast_mark', to=TensorProto.FLOAT))
    init_arr('class8', np.array([1 if i == 8 else 0 for i in range(10)], np.float32).reshape(1, 10, 1, 1))
    nodes.append(helper.make_node('Mul', ['input', 'not_mark'], ['base'], 'base'))
    nodes.append(helper.make_node('Mul', ['mark_f', 'class8'], ['add8'], 'add8'))
    nodes.append(helper.make_node('Add', ['base', 'add8'], ['output'], 'output_add'))
    graph = helper.make_graph(nodes, 'task341_static_bridge', inputs, outputs, inits)
    model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid('', 13)])
    model.ir_version = 7
    onnx.checker.check_model(model)
    return model


EXPORTABLE_PREDICTORS = {'task341': ('task341_static_bridge_direct_onnx', simulate_task341_static_bridge)}
SIMULATOR_ONLY_PREDICTORS = {}


def train_family_task(task):
    payload, info = fit_task341_static_bridge(task)
    if info.get('ok'):
        return make_task341_static_bridge_model(payload), info
    return None, {
        'ok': False,
        'trainer': 'identity_fallback_export',
        'model_version': MODEL_VERSION,
        'reason': info.get('reason', 'task341-only notebook: static bridge simulation failed'),
    }


In [8]:
# Correctness matrix for task341-only static bridge run.
# The target is to move every task from no_predictor/identity to visible_correct with a semantic predictor.
if 'task_ids' not in globals():
    task_ids = ['task341']

correctness_rows = []
for task_id in task_ids:
    task = load_task(DATA_DIR, task_id)
    status = 'no_predictor'
    rule_name = None
    export_status = 'none'
    predictor = None

    if task_id in EXPORTABLE_PREDICTORS:
        rule_name, predictor = EXPORTABLE_PREDICTORS[task_id]
        export_status = 'exportable'
        status = 'has_predictor'
    elif task_id in SIMULATOR_ONLY_PREDICTORS:
        rule_name, predictor = SIMULATOR_ONLY_PREDICTORS[task_id]
        export_status = 'simulator_only'
        status = 'has_predictor'
    else:
        rule_name = 'identity_baseline'
        predictor = identity_grid_predictor
        export_status = 'identity_fallback'
        status = 'identity_baseline'

    summary = evaluate_predictor_on_task(task, predictor)
    if status == 'has_predictor':
        status = 'visible_correct' if summary['right'] == summary['total'] else 'partial'
    elif status == 'identity_baseline' and summary['right'] == summary['total']:
        status = 'identity_correct'

    correctness_rows.append({
        'task_id': task_id,
        'status': status,
        'rule_name': rule_name,
        'export_status': export_status,
        'right': summary['right'],
        'total': summary['total'],
        'accuracy': summary['accuracy'],
        'first_wrong': summary['first_wrong'],
        'train_right': summary['train_right'],
        'train_total': summary['train_total'],
        'test_right': summary['test_right'],
        'test_total': summary['test_total'],
        'arc_gen_right': summary['arc-gen_right'],
        'arc_gen_total': summary['arc-gen_total'],
    })

correctness_df = pd.DataFrame(correctness_rows)
display(correctness_df)
display(correctness_df['status'].value_counts().rename_axis('status').reset_index(name='count'))
display(correctness_df['export_status'].value_counts().rename_axis('export_status').reset_index(name='count'))

unsolved_df = correctness_df[~correctness_df['status'].isin(['visible_correct', 'identity_correct'])].copy()
print('visible-correct tasks:', int((correctness_df['status'] == 'visible_correct').sum()), '/', len(correctness_df))
print('unsolved tasks:', len(unsolved_df))
display(unsolved_df[['task_id', 'status', 'right', 'total', 'accuracy', 'first_wrong']])


,task_id,status,rule_name,export_status,right,total,accuracy,first_wrong,train_right,train_total,test_right,test_total,arc_gen_right,arc_gen_total
0,task341,visible_correct,task341_static_bridge_direct_onnx,exportable,266,266,1.0,None,3,3,1,1,262,262


,status,count
0,visible_correct,1


,export_status,count
0,exportable,1


visible-correct tasks: 1 / 1
unsolved tasks: 0


,task_id,status,right,total,accuracy,first_wrong


In [9]:
# Build one model file for task341-only static bridge run.
# The build uses exportable semantic models where available and identity fallback otherwise.
import shutil

if 'task_ids' not in globals():
    task_ids = ['task341']

# Clear stale models from earlier runs before creating this scoped zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

print('pre-build task ids:', task_ids)
rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=True,
    validate=False,
    task_ids_override=task_ids,
)

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0
print('selected task341-only tasks:', len(task_ids))
print('models saved:', saved_count)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

expected_names = {f'{task_id}.onnx' for task_id in task_ids}
actual_names = {path.name for path in OUT_DIR.glob('task*.onnx')}
print('missing models:', sorted(expected_names - actual_names))
print('extra models:', sorted(actual_names - expected_names))
assert not (expected_names - actual_names), 'missing scoped task models'
assert not (actual_names - expected_names), 'found stale or out-of-scope task models'

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)


pre-build task ids: ['task341']


,task_id,saved,path,ok,trainer,model_version,semantic_kind,visible_right,visible_total,first_wrong,estimated_model_size_bytes
0,task341,True,/kaggle/working/working_submission/fill_enclos...,True,task341_static_bridge_direct_onnx,fill-additive-nonlocal-1color-v0.75-task341-st...,eroded_horizontal_vertical_nonzero_bridge_mark...,266,266,None,9020


selected task341-only tasks: 1
models saved: 1


,trainer,count
0,task341_static_bridge_direct_onnx,1


missing models: []
extra models: []
family zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task341_static_bridge_v75/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [10]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'subtype': SUBTYPE,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'task_ids': task_ids,
    'out_dir': str(OUT_DIR),
    'strategy': 'task341-only direct ONNX eroded horizontal/vertical static bridge marker',
    'exportable_predictors': sorted(EXPORTABLE_PREDICTORS.keys()) if 'EXPORTABLE_PREDICTORS' in globals() else [],
    'simulator_only_predictors': sorted(SIMULATOR_ONLY_PREDICTORS.keys()) if 'SIMULATOR_ONLY_PREDICTORS' in globals() else [],
}
run_manifest


{'family': 'fill_enclosed_regions',
 'subtype': 'nonlocal_1color',
 'model_version': 'fill-additive-nonlocal-1color-v0.75-task341-static-bridge',
 'task_count': 1,
 'task_ids': ['task341'],
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task341_static_bridge_v75',
 'strategy': 'task341-only direct ONNX eroded horizontal/vertical static bridge marker',
 'exportable_predictors': ['task341'],
 'simulator_only_predictors': []}

In [11]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
# 'path' in rows: '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task255_static_onnx_builder_v2_v67/task255.onnx'


validate_rows = []

for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

print(pd.DataFrame(validate_rows))
print("input and expected output: ", summary['first_wrong_expected'])
print("actual model output: ", summary['first_wrong_actual'])

len of examples:  266
example 10
example 20
example 30
example 40
example 50
example 60
example 70
example 80
example 90
example 100
example 110
example 120
example 130
example 140
example 150
example 160
example 170
example 180
example 190
example 200
example 210
example 220
example 230
example 240
example 250
example 260
   task_id  right  wrong
0  task341    266      0
input and expected output:  None
actual model output:  None


In [12]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task341,fill-additive-nonlocal-1color-v0.75-task341-st...,9055,1833,30,"{""Add"": 1, ""And"": 5, ""Cast"": 4, ""Conv"": 3, ""Eq...",99000,117000,3,3,1.0,1,1,1.0,262,262,1.0,266,266,1.0


In [13]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task341_static_bridge_v75/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.75-task341-static-bridge_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color_task341_static_bridge_v75/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.75-task341-static-bridge_manifest.json
